# Paramètre de projet 

In [0]:
# Paramètre : catalog cible (dev aujourd'hui, prod via le déploiement plus tard)
dbutils.widgets.text("CATALOG", "dev", "Catalog cible")
CATALOG = dbutils.widgets.get("CATALOG").strip()
if not CATALOG:
    raise ValueError("Le paramètre CATALOG est vide")
print(f"Catalog cible : {CATALOG}")

# Configuration de projet 

In [0]:
OWNER = "grp-data-admins"   # même propriétaire que les catalogs

SCHEMAS = {
    "landing": "Fichiers bruts deposes par les ingestions (equivalent du conteneur landing)",
    "bronze":  "Tables Delta brutes, une par table source, prefixees par la source",
}

VOLUMES = {
    ("landing", "files"):       "Fichiers bruts : <source>/<aaaa>/<mm>/<jj>/<table>/",
    ("landing", "dlt_staging"): "Zone de transit dlthub pour le chargement direct en bronze",
}

# Création de Schema

In [0]:
for schema, comment in SCHEMAS.items():
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{schema}` COMMENT '{comment}'")
    print(f"✅ Schéma {CATALOG}.{schema}")

# Création de volumes

In [0]:
for (schema, volume), comment in VOLUMES.items():
    spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`{schema}`.`{volume}` COMMENT '{comment}'")
    print(f"✅ Volume {CATALOG}.{schema}.{volume}")

# Propriétaire = groupe admins

In [0]:
for (schema, volume) in VOLUMES:
    spark.sql(f"ALTER VOLUME `{CATALOG}`.`{schema}`.`{volume}` OWNER TO `{OWNER}`")
for schema in SCHEMAS:
    spark.sql(f"ALTER SCHEMA `{CATALOG}`.`{schema}` OWNER TO `{OWNER}`")
print(f"✅ Propriétaire : {OWNER}")

# Vérification et test d’écriture dans la landing
#### on contrôle que les cellules 3 et 4 ont bien fonctionné, puis fait un essai réel dans la landing.
#### on liste les schémas du catalog dev. On doit y voir landing et bronze, et aussi ceux qui existaient déjà : default, information_schema, test_droits.
#### il faut prouver qu’on peut réellement y déposer des fichiers. On simule donc ce que fera dlthub.
#### On prépare le chemin d’un fichier de test. Avec CATALOG = dev, cela donne /Volumes/dev/landing/files/_test_init/ok.txt. Le dossier _test_init commence par _ pour signaler un objet technique ou temporaire.
#### 

In [0]:

display(spark.sql(f"SHOW SCHEMAS IN `{CATALOG}`"))
display(spark.sql(f"SHOW VOLUMES IN `{CATALOG}`.`landing`"))

# test d’écriture, lecture et suppression dans la landing
# On prépare le chemin d’un fichier de test. Avec CATALOG = dev, cela donne /Volumes/dev/landing/files/_test_init/ok.txt

test_path = f"/Volumes/{CATALOG}/landing/files/_test_init/ok.txt"

# on crée le fichier ok.txt avec le texte « test ecriture landing ». overwrite=True permet d’écraser le fichier s’il existe déjà,
dbutils.fs.put(test_path, "test ecriture landing", overwrite=True)

# head lit le début du fichier. On doit voir s’afficher Lecture : test ecriture landing.
print("Lecture :", dbutils.fs.head(test_path))

# Suppression : on efface le dossier de test et son contenu (recurse=True), pour laisser la landing propre.
dbutils.fs.rm(f"/Volumes/{CATALOG}/landing/files/_test_init", recurse=True)
print("✅ Écriture / lecture / suppression OK dans la landing")